In [150]:
# spark.stop()

In [151]:
import os
from pyspark.sql import SparkSession, types as t, functions as F
from pyspark.sql.types import StringType, FloatType, IntegerType

# https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar

spark = (
    SparkSession
    .builder
    # .master("spark://spark-master:7077")
    .appName("Testing Transformations")
    .config("spark.jars", "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar") # GCS Connector
    .getOrCreate()
)

# Google Cloud Service Account Credentials
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile",os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))

spark

In [152]:
bucket='gs://zoomcamp-454219-ade-pipeline/data/pq/'
# year = 2025
# event = "drug-event-part-1-of-34.parquet" # 2025
year = 2004
event = "drug-event-part-1-of-20.parquet" # 2004

df = (
    spark
    .read
    .parquet(bucket+f'patient/{year}/{event}')
    )
print(f"Count: {df.count()}")
# print(df.printSchema())
# patient.show()

Count: 12000


In [153]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType

class Patient:
    columns = [
        'patientid',
        'age_group',
        'sex',
        'weight',
        'expedited_process',
        'primarysourcecountry',
        'occurcountry',
        'report_type',
        'receipt_date',
        'receive_date',
        'safetyreportid',
        'transmission_date',
        'age_years',
        'serious_type'
    ]
    
    def __init__(self, df):
        self.df = df

    def clean_date_column(self, col_name):
        
        # Fix missing days or months
        temp_df = self.df.withColumn(
            col_name,
            (
                F
                .when(F.length(col_name) == 4, F.concat(col_name,F.lit("0101")))
                .when(F.length(col_name) == 6, F.concat(col_name,F.lit("01")))
                .otherwise(F.col(col_name))
            )
        )

        # Set date constraints
        temp_df = temp_df.withColumn(
            col_name,
            (
                F
                .when(F.col(col_name) < F.lit("19000101"), None)
                .otherwise(F.col(col_name))
            )
        )

        # Cast to datetype
        temp_df = temp_df.withColumn(col_name,(F.to_date(col_name,"yyyyMMdd")))

        return temp_df
    
    def get_df(self):
        return self.df

    def cast(self):
        self.df = (
            self.df
            .withColumn("patientid", F.col("patientid").cast(StringType()))
            .withColumn("patientagegroup", F.col("patientagegroup").cast(IntegerType()))
            .withColumn("patientonsetage", F.col("patientonsetage").cast(IntegerType()))
            .withColumn("patientonsetageunit", F.col("patientonsetageunit").cast(IntegerType()))
            .withColumn("patientsex", F.col("patientsex").cast(IntegerType()))
            .withColumn("patientweight", F.col("patientweight").cast(FloatType()))
            .withColumn("serious", F.col("serious").cast(IntegerType()))
            .withColumn("seriousnessdeath", F.col("seriousnessdeath").cast(IntegerType()))
            .withColumn("seriousnesshospitalization", F.col("seriousnesshospitalization").cast(IntegerType()))
            .withColumn("seriousnessdisabling", F.col("seriousnessdisabling").cast(IntegerType()))
            .withColumn("seriousnesslifethreatening", F.col("seriousnesslifethreatening").cast(IntegerType()))
            .withColumn("seriousnessother", F.col("seriousnessother").cast(IntegerType()))
            .withColumn("receivedate", F.col("receivedate").cast(StringType()))
            .withColumn("receiptdate", F.col("receiptdate").cast(StringType()))
            .withColumn("safetyreportid", F.col("safetyreportid").cast(StringType()))

            # Added
            .withColumn("fulfillexpeditecriteria", F.col("fulfillexpeditecriteria").cast(IntegerType()))
            .withColumn("primarysourcecountry", F.col("primarysourcecountry").cast(StringType()))
            .withColumn("occurcountry", F.col("occurcountry").cast(StringType()))
            .withColumn("reporttype", F.col("reporttype").cast(IntegerType()))
            .withColumn("transmissiondate", F.col("transmissiondate").cast(StringType()))
            .withColumn("seriousnesscongenitalanomali", F.col("seriousnesscongenitalanomali").cast(IntegerType()))
        )   

    def transform(self):

        # Normalize patientage to years
        self.df = self.df.withColumn(
            "age_years",
            (
                F
                .when(F.col("patientonsetageunit") == 800, F.col("patientonsetage") * 10)
                .when(F.col("patientonsetageunit") == 801, F.col("patientonsetage") * 1)
                .when(F.col("patientonsetageunit") == 802, F.col("patientonsetage") / 12)
                .when(F.col("patientonsetageunit") == 803, F.col("patientonsetage") / 52.143)
                .when(F.col("patientonsetageunit") == 804, F.col("patientonsetage") / 365.25)
                .when(F.col("patientonsetageunit") == 805, F.col("patientonsetage") / (24 * 365.25))
                .otherwise(None)
            ).cast(FloatType())
        ).drop(
            "patientonsetageunit", "patientonsetage"
        )

        self.df = self.df.withColumn(
            "patientagegroup",
            (
                F
                .when((F.col("patientagegroup") == 1) | (F.col("age_years") * 365.25 < 28), F.lit("Neonate"))
                .when((F.col("patientagegroup") == 2) | ((F.col("age_years") * 365.25 >= 28) & (F.col("age_years") < 1)), F.lit("Infant"))
                .when((F.col("patientagegroup") == 3) | ((F.col("age_years") >= 1) & (F.col("age_years") <= 12)), F.lit("Child"))
                .when((F.col("patientagegroup") == 4) | ((F.col("age_years") >= 13) & (F.col("age_years") <= 17)), F.lit("Adolescent"))
                .when((F.col("patientagegroup") == 5) | ((F.col("age_years") >= 18) & (F.col("age_years") <= 64)), F.lit("Adult"))
                .when((F.col("patientagegroup") == 6) | (F.col("age_years") >= 65), F.lit("Elderly"))
                .otherwise(None)
                )
        ).withColumnRenamed("patientagegroup", "age_group")

        self.df = self.df.withColumn(
            "patientsex",
            (
                F
                .when(F.col("patientsex") == 1, F.lit("Male"))
                .when(F.col("patientsex") == 2, F.lit("Female"))
                .otherwise(None)
            ).cast(StringType())
        ).withColumnRenamed("patientsex", "sex")

        self.df = self.df.withColumn(
            "patientweight",
            (
                F
                .when(
                    F.col("patientweight").rlike(r"^\d+(\.\d+)?$"),
                    F.col("patientweight").cast(FloatType()))
                .otherwise(None)
                )
        ).withColumnRenamed("patientweight", "weight")

        self.df = self.df.withColumn(
            "serious_type",
            F.when(F.col("serious") == 1,
                F.coalesce(
                    F.when(F.col("seriousnessdeath") == 1, F.lit("Death")),
                    F.when(F.col("seriousnesscongenitalanomali") == 1, F.lit("Congenitalanomali")),
                    F.when(F.col("seriousnessdisabling") == 1, F.lit("Disabling")),
                    F.when(F.col("seriousnesshospitalization") == 1, F.lit("Hospitalization")),
                    F.when(F.col("seriousnesslifethreatening") == 1, F.lit("Lifethreatening")),
                    F.when(F.col("seriousnessother") == 1, F.lit("Other")),
                )
            ).otherwise(None)
        ).drop(
                "serious",
                "seriousnessdeath",
                "seriousnesscongenitalanomali",
                "seriousnessdisabling",
                "seriousnesshospitalization",
                "seriousnesslifethreatening",
                "seriousnessother"
            )
        
        self.df = self.clean_date_column("receivedate").withColumnRenamed("receivedate","receive_date")
        
        self.df = self.clean_date_column("receiptdate").withColumnRenamed("receiptdate","receipt_date")

        self.df = self.df.withColumn(
            "fulfillexpeditecriteria",
            (
                F
                .when(F.col('fulfillexpeditecriteria') == 1, "Yes")
                .when(F.col('fulfillexpeditecriteria') == 2, "No")
                .otherwise(None)
                )
            ).withColumnRenamed("fulfillexpeditecriteria","expedited_process")

        self.df = self.df.withColumn(
            "reporttype", 
            (
                F
                .when(F.col("reporttype") == 1, F.lit("Spontaneous"))
                .when(F.col("reporttype") == 2, F.lit("Report from study"))
                .when(F.col("reporttype") == 3, F.lit("Other"))
                .when(F.col("reporttype") == 4, F.lit("Unknown"))
                .otherwise(None)
            )
        ).withColumnRenamed("reporttype","report_type")

        self.df = self.clean_date_column("transmissiondate").withColumnRenamed("transmissiondate","transmission_date")

        # Handle null
        self.handle_null()

    def handle_null(self):
        """
        Must handle null values for all the fields not just the prone ones
        Case:
        
        - Entire field could be null
        - Entire row could be null
        - Field contains partial null
        - Row contains partial null
        """
        # TODO
        # age_group                 - Fill with 'Unspecified'
        # sex                       - Fill with 'Unknown'
        # weight                    - Fill with -1
        # expedited_process         - Fill with False
        # primarysourcecountry      - Fill with 'ZZ'
        # occurcountry              - Fill with 'ZZ'
        # report_type               - Fill with 'Unknown'
        # receipt_date              - Leave as it is
        # receive_date              - Leave as it is
        # safetyreportid            - Leave as it is
        # transmission_date         - Leave as it is
        # age_years                 - Replace NaN with -1.0
        # serious_type              - Replace with 'Not Serious'
        
        fillna_dict = {
            'age_group' : 'Unspecified',
            'sex' : 'Unknown',
            'weight' : -1,
            'expedited_process' : 'Unknown',
            'primarysourcecountry' : 'ZZ',
            'occurcountry' : 'ZZ',
            'report_type' : 'Unknown',
            'age_years' : -1.0,
            'serious_type' : 'Not Serious',
        }

        self.df = self.df.fillna(fillna_dict)

    def get_null_count(self):
        return self.df.select([
            F.sum(
                F.when(
                    F.col(c).isNull(), 1
                ).otherwise(0)
            ).alias(c)
            for c in self.df.columns
        ]).first().asDict()
    
    def get_count(self):
        return self.df.count()


In [154]:
from time import time

class Metrics:
    def __init__(self, schema):

        self.schema = schema
        self.start_time = time()
        self.total_records = 0
        self.null_count = {}
        self.null_ratio = {}

    def reset(self):
        self.start_time = time()
        self.total_records = 0
        self.null_count = {}
        self.null_ratio = {}

    def update(self, obj):

        # Update total records
        self.total_records += obj.get_count()

        # Update null count
        for k,v in obj.get_null_count().items():
            self.null_count[k] = self.null_count.get(k, 0) + v

    def _null_ratio(self):
        for k,v in self.null_count.items():
            try:
                self.null_ratio[k] = float(v / self.total_records)
            except ZeroDivisionError:
                self.null_ratio[k] = float(0.0)

    def publish(self, ):
        # Select what to publish only
        self._null_ratio()

        # Processing time
        duration = time() - self.start_time

        # Update Gauge
        pass

    def close(self):
        # self.reset()
        # delete_from_gateway(gateway=self.gateway,job=self.job)
        pass

In [155]:
from pprint import pprint
metrics = Metrics(schema='patient')

p = Patient(df)
p.cast()
p.transform()

print(p.get_count())
pprint(p.get_null_count())

12000


{'age_group': 0,
 'age_years': 0,
 'expedited_process': 0,
 'occurcountry': 0,
 'patientid': 0,
 'primarysourcecountry': 0,
 'receipt_date': 0,
 'receive_date': 0,
 'report_type': 0,
 'safetyreportid': 0,
 'serious_type': 0,
 'sex': 0,
 'transmission_date': 0,
 'weight': 0}


In [156]:
metrics.update(obj=p)
metrics._null_ratio()
metrics.null_ratio

{'patientid': 0.0,
 'age_group': 0.0,
 'sex': 0.0,
 'weight': 0.0,
 'expedited_process': 0.0,
 'primarysourcecountry': 0.0,
 'occurcountry': 0.0,
 'report_type': 0.0,
 'receipt_date': 0.0,
 'receive_date': 0.0,
 'safetyreportid': 0.0,
 'transmission_date': 0.0,
 'age_years': 0.0,
 'serious_type': 0.0}

| **Stage**      | **Age Range**                               |
| -------------- | ------------------------------------------- |
| **Neonate**    | Birth to **28 days**                        |
| **Infant**     | **1 month to 1 year**                       |
| **Child**      | **1 year to \~12 years**                    |
| **Adolescent** | **13 to 17 years** (sometimes up to 19)     |
| **Adult**      | **18 to 64 years** (sometimes starts at 20) |
| **Elderly**    | **65 years and older**                      |
